# Ranker exploration (Raasta B, component 2)

Reads the held-out ranker tables (`ranker.py --table`) and the out-of-fold scores (`ranker.py --cv`) and asks where the learned ranker beats v17's hand weights, and where it does not. Numbers in `LEARNING.md` entries 4-5 and `RESULTS.md` section 33 come from the job scripts, not from this notebook; this is for looking.

Run from JupyterLab at the repo root: `.venv/bin/jupyter lab`.

In [ ]:
import os, sys, glob
import numpy as np
import matplotlib.pyplot as plt
HERE = os.path.abspath('..')        # MoleculeID-from-Mass-Spectra, this notebook's parent
sys.path.insert(0, HERE)
os.environ.setdefault('CASMI_DATA', os.path.join(HERE, 'data'))
import features, ranker
D = os.environ['CASMI_DATA']
TABLES = sorted(glob.glob(f'{D}/rank_tables/splits_bigord_*.npz'))[-1:] + \
         sorted(glob.glob(f'{D}/rank_tables/splits_big_*.npz'))[-1:]
T = ranker.load_tables(TABLES)
F = features.FEATURES
print(TABLES, len(T['y']), 'rows')

## Out-of-fold scores

The newest OOF file of each model run on exactly these tables.

In [ ]:
def newest(model, drop):
    for p in sorted(glob.glob(f'{D}/rank_oof/{model}_drop-{drop}_*.npz'), reverse=True):
        z = np.load(p)
        if [str(t) for t in z['tables']] == TABLES:
            return p, z['oof']
    return None, None
OOF = {}
for model in ('gbm', 'linear', 'lambdarank'):
    for drop in ('frag+formula', 'frag', 'none'):
        p, s = newest(model, drop)
        if p:
            OOF[f'{model} ({drop})'] = s
v17 = T['X'][:, F.index('v17')]
for name, s in OOF.items():
    print(f'{name:28s}', {k: round(v, 4) for k, v in ranker.mrr_by(T, s).items()})
print(f"{'v17':28s}", {k: round(v, 4) for k, v in ranker.mrr_by(T, v17).items()})

## Answer against the other candidates, per feature

Big covered class 3: what separates the answer from the isomers around it.

In [ ]:
cov3 = (T['split'] == 'splits_big') & (T['qclass'] == 3)
rows = cov3[T['qid']]
show = ['v17_gap', 'fp', 'fp_rank', 'shift3', 'prior', 'spec', 'frag', 'formula']
fig, ax = plt.subplots(2, 4, figsize=(16, 6))
for a, f in zip(ax.ravel(), show):
    x = T['X'][rows, F.index(f)]
    y = T['y'][rows].astype(bool)
    bins = np.linspace(np.percentile(x, 1), np.percentile(x, 99) + 1e-9, 40)
    a.hist(x[~y], bins, density=True, alpha=0.5, label='other candidates')
    a.hist(x[y], bins, density=True, alpha=0.5, label='answer')
    a.set_title(f'{f}  (pool AUC {ranker.pool_auc(ranker.subset(T, cov3), f):.3f})')
ax[0, 0].legend()
plt.tight_layout()

## Where the answer moved

v17 rank against the ranker's rank, answers inside the top 100 only.

In [ ]:
MODEL = 'lambdarank (frag+formula)' if 'lambdarank (frag+formula)' in OOF else next(iter(OOF))
def answer_rank(s):
    rr = ranker.rr_by_query(T, s)
    return np.where(rr > 0, 1 / np.maximum(rr, 1e-9), np.nan)
r0, r1 = answer_rank(v17), answer_rank(OOF[MODEL])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (name, m) in zip(ax, [('big covered c3', (T['split'] == 'splits_big') & (T['qclass'] == 3)),
                             ('bigord non-env c3', (T['split'] == 'splits_bigord') & (T['qclass'] == 3) & ~T['env'])]):
    j = np.random.default_rng(0).uniform(-0.2, 0.2, (2, m.sum()))
    a.scatter(np.nan_to_num(r0[m], nan=30) + j[0], np.nan_to_num(r1[m], nan=30) + j[1], s=6, alpha=0.4)
    a.plot([1, 30], [1, 30], 'k--', lw=0.8)
    a.set(xscale='log', yscale='log', xlabel='v17 rank (30 = outside the top 25)',
          ylabel=f'{MODEL} rank', title=name)
plt.tight_layout()

## The ten biggest moves either way

Top-5 candidates of each, with the features the ranker saw.

In [ ]:
cols = ['v17', 'fp', 'shift3', 'prior', 'spec', 'frag']
def show_query(q, s):
    rows = np.flatnonzero(T['qid'] == q)
    o = rows[np.argsort(-s[rows], kind='stable')][:5]
    print(f"  answer {T['answer'][q].decode()}")
    for i in o:
        mark = '*' if T['y'][i] else ' '
        vals = '  '.join(f'{c} {T["X"][i, F.index(c)]:.3f}' for c in cols)
        print(f"  {mark} {T['smiles'][i].decode()[:60]:60s} {vals}")
m = (T['qclass'] == 3) & np.isfinite(r0) & np.isfinite(r1)
gain = np.where(m, np.log(r0) - np.log(r1), 0.0)
for title, qs in (('moved up', np.argsort(-gain)[:10]), ('moved down', np.argsort(gain)[:10])):
    print(f'=== {title} ({MODEL})')
    for q in qs:
        print(f"{T['split'][q]} {T['qkey'][q].decode()}: v17 rank {r0[q]:.0f} -> {r1[q]:.0f}")
        show_query(q, OOF[MODEL])

## frag vs formula

`formula` asks whether a peak's formula fits inside the candidate's formula, so isomers, which share one formula, get the same value. `frag` (MetFrag-lite) breaks the candidate's own bonds, one or two at a time, and asks how much of the spectrum's intensity sits at a fragment ion, so isomers can differ. Big covered class 3, answer against the other candidates.

In [ ]:
show = ['frag', 'frag_mean', 'frag_gap', 'formula']
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
y = T['y'][rows].astype(bool)
for a, f in zip(ax, show):
    x = T['X'][rows, F.index(f)]
    bins = np.linspace(np.percentile(x, 1), np.percentile(x, 99) + 1e-9, 40)
    a.hist(x[~y], bins, density=True, alpha=0.5, label='other candidates')
    a.hist(x[y], bins, density=True, alpha=0.5, label='answer')
    a.set_title(f'{f}  (pool AUC {ranker.pool_auc(ranker.subset(T, cov3), f):.3f})')
ax[0].legend()
plt.tight_layout()
# The answer against its exact isomers only (same molecular formula), where formula cannot help.
import formula
def hill(s):
    c = formula.counts(s.decode())
    return None if c is None else c[0]
idx = np.flatnonzero(rows)
H = np.array([hill(s) for s in T['smiles'][idx]], dtype=object)
iso = np.zeros(len(T['y']), bool)
for qq in np.unique(T['qid'][idx][T['y'][idx].astype(bool)]):
    m = T['qid'][idx] == qq
    a = H[m & T['y'][idx].astype(bool)][0]
    if a is not None:
        iso[idx[m]] = [h == a for h in H[m]]
ISO = dict(T, qid=np.where(iso, T['qid'], -1), y=np.where(iso, T['y'], 0))
n_q = len(np.unique(T['qid'][iso & ~T['y'].astype(bool)]))
print(f'answers with at least one isomer among their candidates: {n_q}, '
      f'{(iso & ~T["y"].astype(bool)).sum()} isomers')
for f in ('frag', 'frag_mean', 'formula', 'fp', 'prior', 'v17'):
    print(f'  {f:10s} AUC against isomers {ranker.pool_auc(ISO, f):.3f}')

## Five molecules MetFrag-lite pulled up

The answer's rank with every feature against the rank with `frag` dropped (one-seed lambdarank runs of the ablation). For each, the spectrum's peaks and which of them the answer's fragments and the no-frag winner's fragments explain.

In [ ]:
import library, formula, metfrag
from analog import ADDUCTS
from search import spectra_of
ix = library.load()
SP = {s: np.load(f'{D}/{s}.npz', allow_pickle=True) for s in ('splits_big', 'splits_bigord')}
def spectra_and_adduct(q):
    sp = SP[str(T['split'][q])]
    r = sp['query'][sp['qkey'] == T['qkey'][q]]
    spectra = spectra_of(ix, r)
    add = [a for a, (_, p) in ADDUCTS.items() if p == bool(ix['ion'][r[0]])]
    mass = formula.counts(ix['smiles'][r[0]].decode())[1]
    return spectra, min(add, key=lambda a: abs(mass + ADDUCTS[a][0] - spectra[0][2]))
def peaks_explained(smiles, mzs, add):
    f = metfrag.fragment_masses(smiles)
    return np.array([metfrag.explain(f, [m], [1.0], add) > 0 for m in mzs])
with_frag, no_frag = newest('lambdarank', 'none')[1], newest('lambdarank', 'frag')[1]
ra, rb = answer_rank(with_frag), answer_rank(no_frag)
m = cov3 & np.isfinite(ra) & np.isfinite(rb)
gain = np.where(m, np.log(rb) - np.log(ra), 0.0)
for q in np.argsort(-gain)[:5]:
    rws = np.flatnonzero(T['qid'] == q)
    ans = rws[T['y'][rws].astype(bool)][0]
    top = rws[np.argmax(no_frag[rws])]
    spectra, add = spectra_and_adduct(q)
    fr = [metfrag.explain(metfrag.fragment_masses(T['smiles'][ans].decode()), mz, it, add) for mz, it, _ in spectra]
    mz, it, pm = spectra[int(np.argmax(fr))]
    ea = peaks_explained(T['smiles'][ans].decode(), mz, add)
    et = peaks_explained(T['smiles'][top].decode(), mz, add)
    print(f"{T['qkey'][q].decode()} {add} precursor {pm:.4f}: answer rank {rb[q]:.0f} without frag -> {ra[q]:.0f} with")
    for name, i, e in (('answer', ans, ea), ('no-frag top', top, et)):
        print(f"  {name:12s} frag {T['X'][i, F.index('frag')]:.3f}  formula {T['X'][i, F.index('formula')]:.3f}  "
              f"{T['smiles'][i].decode()[:70]}")
    o = np.argsort(-it)[:12]
    print('  peak m/z   intensity  answer  no-frag top')
    for j in sorted(o, key=lambda j: mz[j]):
        print(f"  {mz[j]:9.4f}  {it[j]:8.3f}   {'x' if ea[j] else '.':^6s}  {'x' if et[j] else '.':^6s}")